In [1]:
%%capture
!pip install pip3-autoremove
!pip-autoremove torch torchvision torchaudio -y
!pip install torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu121
!pip install unsloth

In [2]:
import torch
print(torch.cuda.device_count(), "GPUs available.")


2 GPUs available.


In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.
hf_token=''
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit", 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    token = hf_token
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2024.12.11: Fast Qwen2 patching. Transformers: 4.47.1.
   \\   /|    GPU: Tesla T4. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.28.post3. FA2 = False]
 "-____-"     Free Apache license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.51k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [4]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 8, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    use_gradient_checkpointing = True,
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2024.12.11 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


<a name="Data"></a>
### Data Prep
We now use the Alpaca dataset from [yahma](https://huggingface.co/datasets/yahma/alpaca-cleaned), which is a filtered version of 52K of the original [Alpaca dataset](https://crfm.stanford.edu/2023/03/13/alpaca.html). You can replace this code section with your own data prep.

**[NOTE]** To train only on completions (ignoring the user's input) read TRL's docs [here](https://huggingface.co/docs/trl/sft_trainer#train-on-completions-only).

**[NOTE]** Remember to add the **EOS_TOKEN** to the tokenized output!! Otherwise you'll get infinite generations!

If you want to use the `ChatML` template for ShareGPT datasets, try our conversational [notebook](https://colab.research.google.com/drive/1Aau3lgPzeZKQ-98h69CCu1UJcvIBLmy2?usp=sharing).

For text completions like novel writing, try this [notebook](https://colab.research.google.com/drive/1ef-tab5bhkvWmBOObepl1WgJvfvSzn5Q?usp=sharing).

In [5]:
import pandas as pd 
df=pd.read_csv('/kaggle/input/raid-arabic-stories/merged_results_final.csv')



df=df[df["متوافقة مع القيم الإسلامية؟"] == True]



df.drop(columns=["متوافقة مع القيم الإسلامية؟", "السبب"],inplace=True)
df=df[~df["prompt"].str.contains(
    "اقرأ القصة التالية بعناية. مهمتك هي تحديد ما إذا كانت القصة تتوافق مع القيم الإسلامية الأساسية"
)]
df.head()




,prompt,قصة,عنوان,الفئة العمرية,القيمة المميزة
1200,اكتب قصة تعليمية (3-5 فقرات) موجهة للأطفال الص...,في مدينة بروكلين الصاخبة، كان هناك عرض خاص لفي...,معالجة العلم والمرح,6-8,التعاون
1201,اكتب قصة تعليمية (من 3 إلى 5 فقرات) موجهة للأط...,في بلدة صاخبة مليئة بالمخلوقات الغريبة التي تس...,كودي وبايت في رحلة الاستكشاف,6-8,التعاون
1202,اكتب قصة تعليمية (3-5 فقرات) موجهة للأطفال الص...,ذات مرة، في أرض مليئة بالكتب والقصص، عاشت كلمت...,إن وجامب وتعليم اللغة,6-8,التعلم
1203,اكتب قصة تعليمية (3-5 فقرات) موجهة للأطفال الص...,ذات مرة، في بلدة صغيرة تسمى هارمونيفيل، عاش ثل...,بلوك بوكس والحقيقة,6-8,الصدق
1204,اكتب قصة تعليمية (3-5 فقرات) موجهة للأطفال الص...,كان تيمي الصغير يحب اللعب في الخارج والتحديق ف...,رحلة تيمي وبيلي إلى عالم الجينات,6-8,العلم _ والرجاء


In [6]:
enhanced_prompt = """
أنت نموذج ذكاء اصطناعي متخصص في إنشاء قصص موجهة للأطفال تعزز القيم الإسلامية. تتمثل مهمتك في كتابة قصة جذابة ومناسبة للفئة العمرية المحددة، مع التركيز على القيمة الأخلاقية المطلوبة. يجب أن تكون القصة مكتوبة بلغة عربية سهلة ومبسطة للأطفال، وتتضمن شخصيات وأحداث تُشجع الأطفال على تبني هذه القيمة في حياتهم اليومية.

### التعليمات:
{prompt}
الفئة العمرية المستهدفة: {age_category}.
القيمة الأخلاقية المطلوبة: {moral}.

### الإجابة:
### العنوان:
{title}

### القصة:
{generated_response}
"""

# Function to format the dataset
def formatting_prompts_func(row):
    # Extract values from the row
    prompt = row['prompt']
    title = row["عنوان"]
    story = row["قصة"]
    age_category = row["الفئة العمرية"]
    moral = row["القيمة المميزة"]
 
    text = enhanced_prompt.format(
        prompt=prompt,
        title=title,
        age_category=age_category,
        moral=moral,  # Add 'reason' here to ensure no KeyError
        generated_response=story
    )
    return text

# Apply the function to the DataFrame to format the text for each row
df['formatted_prompt'] = df.apply(formatting_prompts_func, axis=1)

# Display the result for the first row
df[['formatted_prompt']].head().iloc[0][0]


<ipython-input-6-a0bc038a8d73>:39: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  df[['formatted_prompt']].head().iloc[0][0]


'\nأنت نموذج ذكاء اصطناعي متخصص في إنشاء قصص موجهة للأطفال تعزز القيم الإسلامية. تتمثل مهمتك في كتابة قصة جذابة ومناسبة للفئة العمرية المحددة، مع التركيز على القيمة الأخلاقية المطلوبة. يجب أن تكون القصة مكتوبة بلغة عربية سهلة ومبسطة للأطفال، وتتضمن شخصيات وأحداث تُشجع الأطفال على تبني هذه القيمة في حياتهم اليومية.\n\n### التعليمات:\nاكتب قصة تعليمية (3-5 فقرات) موجهة للأطفال الصغار باستخدام كلمات بسيطة. يجب أن تكون القصة مستوحاة من مقتطف النص التالي:\n\n"لقد شاهدت هذا الفيلم بالصدفة في عرض في بروكلين - من الصعب وصف الحبكة؛ فهو يحتوي على الكثير من الشخصيات الغريبة، ولكن دعنا نقول فقط إنني سأجد صعوبة في اختيار الشخصية التي تجعلني أضحك بشدة، ولن أعرف من أين أبدأ. حتى الأدوار الهامشية مكتوبة بشكل جيد وممثلة بشكل جيد.\n\nهناك العديد من اللمسات الصغيرة التي تجعله فريدًا وممتعًا للغاية، فهو يحتوي على بعض "الأجهزة" التي تظهر وتضيف طبقة أخرى من المرح. إنه منعش للمشاهدة؛ وليس بعض الأشياء المعاد تدويرها التي رأيتها مرات عديدة من قبل. إذا تمكن هذا الفيلم من الوصول إلى جمهور أوسع، فأنا متأكد من أنه

In [7]:
from datasets import Dataset
dataset = Dataset.from_pandas(df[['formatted_prompt']])

# Display the first few entries of the dataset
print(dataset[0])

{'formatted_prompt': '\nأنت نموذج ذكاء اصطناعي متخصص في إنشاء قصص موجهة للأطفال تعزز القيم الإسلامية. تتمثل مهمتك في كتابة قصة جذابة ومناسبة للفئة العمرية المحددة، مع التركيز على القيمة الأخلاقية المطلوبة. يجب أن تكون القصة مكتوبة بلغة عربية سهلة ومبسطة للأطفال، وتتضمن شخصيات وأحداث تُشجع الأطفال على تبني هذه القيمة في حياتهم اليومية.\n\n### التعليمات:\nاكتب قصة تعليمية (3-5 فقرات) موجهة للأطفال الصغار باستخدام كلمات بسيطة. يجب أن تكون القصة مستوحاة من مقتطف النص التالي:\n\n"لقد شاهدت هذا الفيلم بالصدفة في عرض في بروكلين - من الصعب وصف الحبكة؛ فهو يحتوي على الكثير من الشخصيات الغريبة، ولكن دعنا نقول فقط إنني سأجد صعوبة في اختيار الشخصية التي تجعلني أضحك بشدة، ولن أعرف من أين أبدأ. حتى الأدوار الهامشية مكتوبة بشكل جيد وممثلة بشكل جيد.\n\nهناك العديد من اللمسات الصغيرة التي تجعله فريدًا وممتعًا للغاية، فهو يحتوي على بعض "الأجهزة" التي تظهر وتضيف طبقة أخرى من المرح. إنه منعش للمشاهدة؛ وليس بعض الأشياء المعاد تدويرها التي رأيتها مرات عديدة من قبل. إذا تمكن هذا الفيلم من الوصول إلى جمهور أو

<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`. We also support TRL's `DPOTrainer`!

In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "formatted_prompt",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 8 ,
        gradient_accumulation_steps = 1,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Map (num_proc=2):   0%|          | 0/8112 [00:00<?, ? examples/s]

In [9]:
#@title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.741 GB.
5.744 GB of memory reserved.


In [10]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs = 1
   \\   /|    Num examples = 8,112 | Num Epochs = 1
O^O/ \_/ \    Batch size per device = 8 | Gradient Accumulation steps = 1
\        /    Total batch size = 8 | Total steps = 60
 "-____-"     Number of trainable parameters = 20,185,088


Step,Training Loss
1,2.015000
2,2.027900
3,1.972600
4,1.967100
5,1.867800
6,1.808300
7,1.776800
8,1.805900
9,1.730700
10,1.653800


In [11]:
#@title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory         /max_memory*100, 3)
lora_percentage = round(used_memory_for_lora/max_memory*100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training.")
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

2824.0243 seconds used for training.
47.07 minutes used for training.
Peak reserved memory = 12.467 GB.
Peak reserved memory for training = 6.723 GB.
Peak reserved memory % of max memory = 84.574 %.
Peak reserved memory for training % of max memory = 45.607 %.


<a name="Inference"></a>
### Inference
Let's run the model! You can change the instruction and input - leave the output blank!

In [12]:
# alpaca_prompt = Copied from above
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
inputs = tokenizer(
[
   """
فيما يلي تعليمات تصف مهمة، مقترنة بسياق إضافي. اكتب إجابة تُكمل الطلب بشكل مناسب.

### التعليمات:
اكتب قصة تعليمية (3-5 فقرات) تستهدف الأطفال الصغار باستخدام كلمات بسيطة. يجب أن تكون القصة مستوحاة من 
الفئة العمرية: 6-8.
القيمة: التعاون.

يجب أن تكون القصة موجهة للأطفال ضمن الفئة العمرية: 6-8.
وينبغي أن تركز على القيمة التالية: التعاون.

### الإجابة:
"""
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
tokenizer.batch_decode(outputs)

#
# You can also use a `TextStreamer` for continuous inference -
#so you can see the generation token by token, instead of waiting the whole time!
#
#from transformers import TextStreamer
#text_streamer = TextStreamer(tokenizer)
#_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

['\nفيما يلي تعليمات تصف مهمة، مقترنة بسياق إضافي. اكتب إجابة تُكمل الطلب بشكل مناسب.\n\n### التعليمات:\nاكتب قصة تعليمية (3-5 فقرات) تستهدف الأطفال الصغار باستخدام كلمات بسيطة. يجب أن تكون القصة مستوحاة من \nالفئة العمرية: 6-8.\nالقيمة: التعاون.\n\nيجب أن تكون القصة موجهة للأطفال ضمن الفئة العمرية: 6-8.\nوينبغي أن تركز على القيمة التالية: التعاون.\n\n### الإجابة:\n### العنوان:\n[لا تدع الأخطاء تعيقنا]@\n\n### القصة:\nذات مرة، في بلدة صغيرة تدعى تيرابول، عاش ثلاثة أصدقاء حميمين يسمون سام وليز وبيكي. كانوا يحبون الل']

<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [13]:
#model.save_pretrained("lora_model") # Local saving
model.push_to_hub("qween7.5-arabic-story-teller-bnb-4bit", token = hf_token) # Online saving

README.md:   0%|          | 0.00/590 [00:00<?, ?B/s]

  0%|          | 0/1 [00:00<?, ?it/s]

adapter_model.safetensors:   0%|          | 0.00/80.8M [00:00<?, ?B/s]

Saved model to https://huggingface.co/qween7.5-arabic-story-teller-bnb-4bit


### Saving to float16 for VLLM

We also support saving to `float16` directly. Select `merged_16bit` for float16 or `merged_4bit` for int4. We also allow `lora` adapters as a fallback. Use `push_to_hub_merged` to upload to your Hugging Face account! You can go to https://huggingface.co/settings/tokens for your personal tokens.

In [14]:

if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_16bit", token = "")

# Merge to 4bit
if False: model.save_pretrained_merged("model", tokenizer, save_method = "merged_4bit",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "merged_4bit", token = "")

# Just LoRA adapters
if False: model.save_pretrained_merged("model", tokenizer, save_method = "lora",)
if False: model.push_to_hub_merged("hf/model", tokenizer, save_method = "lora", token = "")

### GGUF / llama.cpp Conversion
To save to `GGUF` / `llama.cpp`, we support it natively now! We clone `llama.cpp` and we default save it to `q8_0`. We allow all methods like `q4_k_m`. Use `save_pretrained_gguf` for local saving and `push_to_hub_gguf` for uploading to HF.

Some supported quant methods (full list on our [Wiki page](https://github.com/unslothai/unsloth/wiki#gguf-quantization-options)):
* `q8_0` - Fast conversion. High resource use, but generally acceptable.
* `q4_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q4_K.
* `q5_k_m` - Recommended. Uses Q6_K for half of the attention.wv and feed_forward.w2 tensors, else Q5_K.

In [15]:
# Save to 8bit Q8_0
if False: model.save_pretrained_gguf("model", tokenizer,)
if False: model.push_to_hub_gguf("hf/model", tokenizer, token = "")

# Save to 16bit GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "f16")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "f16", token = "")

# Save to q4_k_m GGUF
if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")
if False: model.push_to_hub_gguf("hf/model", tokenizer, quantization_method = "q4_k_m", token = "")

Now, use the `model-unsloth.gguf` file or `model-unsloth-Q4_K_M.gguf` file in `llama.cpp` or a UI based system like `GPT4All`. You can install GPT4All by going [here](https://gpt4all.io/index.html).